In [ ]:
save_dir = repo_root / 'models'
save_dir.mkdir(exist_ok=True)

# Save model weights
model_path = save_dir / 'lstm_entry_exit_weights'
model.save_weights(str(model_path))
print(f'Model saved to: {model_path}')

# Save scaler
import joblib
scaler_path = save_dir / 'lstm_feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved to: {scaler_path}')

# To load:
# model.load_weights(str(model_path))
# scaler = joblib.load(scaler_path)

## 7) Save Model

In [ ]:
# Predictions
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification metrics
from sklearn.metrics import classification_report, confusion_matrix
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['SHORT', 'HOLD', 'LONG']))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Simple backtest
def backtest_lstm(predictions, true_labels, initial_capital=100000):
    """Simple backtest simulator"""
    capital = initial_capital
    position = 0  # 0=flat, 1=long, -1=short
    pnl_series = [0]
    trades = 0
    
    for i in range(1, len(predictions)):
        action = predictions[i]  # 0=SHORT, 1=HOLD, 2=LONG
        actual_move = true_labels[i]  # Actual market direction
        
        # P&L calculation (simplified)
        if position == 1 and actual_move == 2:  # Long & market up
            pnl_series.append(pnl_series[-1] + 100)
        elif position == -1 and actual_move == 0:  # Short & market down
            pnl_series.append(pnl_series[-1] + 100)
        elif position != 0 and actual_move == 1:  # Position but market flat
            pnl_series.append(pnl_series[-1] - 10)  # Small decay
        elif (position == 1 and actual_move == 0) or (position == -1 and actual_move == 2):
            pnl_series.append(pnl_series[-1] - 100)  # Wrong direction
        else:
            pnl_series.append(pnl_series[-1])
        
        # Update position based on prediction
        if action == 2:
            position = 1
            trades += 1
        elif action == 0:
            position = -1
            trades += 1
        else:
            position = 0
    
    return pnl_series, trades

pnl_series, num_trades = backtest_lstm(y_pred, y_test)

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Equity curve
ax1.plot(pnl_series)
ax1.set_title(f'Backtest Equity Curve (Total Trades: {num_trades})')
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Cumulative P&L')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)

# Action distribution
actions, counts = np.unique(y_pred, return_counts=True)
action_names = ['SHORT', 'HOLD', 'LONG']
ax2.bar([action_names[a] for a in actions], counts)
ax2.set_title('Predicted Actions Distribution')
ax2.set_ylabel('Count')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'\nFinal P&L: {pnl_series[-1]:.2f}')
print(f'Total Trades: {num_trades}')
print(f'Avg P&L per trade: {pnl_series[-1]/max(num_trades,1):.2f}')

## 6) Evaluate & Backtest

In [ ]:
# Callbacks
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFinal Val Loss: {history.history["val_loss"][-1]:.4f}')
print(f'Final Val Accuracy: {history.history["val_accuracy"][-1]:.4f}')

## 5) Train Model

In [ ]:
class LSTMTradingModel(Model):
    """
    Pure LSTM for entry/exit prediction
    
    Architecture:
    - LSTM layer 1: 64 units (captures short-term patterns)
    - LSTM layer 2: 32 units (captures longer patterns)
    - Dense: 32 units
    - Output: 3 classes (SHORT, HOLD, LONG)
    """
    def __init__(self, n_features, n_classes=3, lstm1_units=64, lstm2_units=32, dropout=0.2):
        super().__init__()
        self.lstm1 = layers.LSTM(lstm1_units, return_sequences=True, name='lstm1')
        self.dropout1 = layers.Dropout(dropout)
        self.lstm2 = layers.LSTM(lstm2_units, return_sequences=False, name='lstm2')
        self.dropout2 = layers.Dropout(dropout)
        self.dense1 = layers.Dense(32, activation='relu', name='dense1')
        self.dropout3 = layers.Dropout(dropout)
        self.output_layer = layers.Dense(n_classes, activation='softmax', name='output')
    
    def call(self, x, training=False):
        x = self.lstm1(x, training=training)
        x = self.dropout1(x, training=training)
        x = self.lstm2(x, training=training)
        x = self.dropout2(x, training=training)
        x = self.dense1(x)
        x = self.dropout3(x, training=training)
        return self.output_layer(x)

# Build model
n_features = X_train.shape[2]
model = LSTMTradingModel(n_features=n_features)

# Compile
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Build by calling once
dummy = tf.zeros((1, lookback, n_features))
_ = model(dummy)

print('Model built successfully')
model.summary()

## 4) Build LSTM Model

In [ ]:
def create_sequences(data, lookback=30, forecast_horizon=5):
    """
    Create sequences for LSTM training
    
    Label strategy: Predict if price will rise/fall/stay in next N bars
    - Label 0: SHORT (price drops > threshold)
    - Label 1: HOLD (price moves within threshold)
    - Label 2: LONG (price rises > threshold)
    """
    X, y = [], []
    
    for i in range(lookback, len(data) - forecast_horizon):
        # Input: last 'lookback' bars
        X.append(data[i-lookback:i])
        
        # Label: future price movement
        current_price = data[i, feature_cols.index('close')]
        future_price = data[i+forecast_horizon, feature_cols.index('close')]
        price_change_pct = (future_price - current_price) / current_price
        
        # Threshold: 0.2% for intraday
        threshold = 0.002
        if price_change_pct > threshold:
            label = 2  # LONG
        elif price_change_pct < -threshold:
            label = 0  # SHORT
        else:
            label = 1  # HOLD
        
        y.append(label)
    
    return np.array(X), np.array(y)

# Scale features
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df.values)

# Create sequences
lookback = 30
forecast_horizon = 5
X, y = create_sequences(data_scaled, lookback, forecast_horizon)

print(f'Sequences created: X shape = {X.shape}, y shape = {y.shape}')
print(f'Label distribution: SHORT={np.sum(y==0)}, HOLD={np.sum(y==1)}, LONG={np.sum(y==2)}')

# Train/test split (80/20)
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {len(X_train)} samples, Test: {len(X_test)} samples')

## 3) Create Sequences & Labels for Supervised Learning

In [ ]:
processed_dir = repo_root / 'data' / 'processed_data'
df = None

if processed_dir.exists():
    for fn in ['train.csv', 'test.csv']:
        p = processed_dir / fn
        if p.exists():
            try:
                df = pd.read_csv(p, parse_dates=True, index_col=0)
                print(f'Loaded: {fn} - {len(df)} rows, {df.shape[1]} columns')
                break
            except Exception as e:
                print(f'Failed to load {fn}:', e)

# Fallback: synthetic data
if df is None:
    print('Creating synthetic intraday data...')
    np.random.seed(42)
    timestamps = pd.date_range('2025-01-01 09:15', periods=360, freq='1min')
    base = 22000 + np.cumsum(np.random.randn(len(timestamps)) * 2)
    df = pd.DataFrame({
        'open': base + np.random.randn(len(timestamps)),
        'high': base + np.abs(np.random.randn(len(timestamps)) * 3),
        'low': base - np.abs(np.random.randn(len(timestamps)) * 3),
        'close': base,
        'volume': np.random.randint(500, 5000, len(timestamps))
    }, index=timestamps)
    print(f'Created synthetic data: {len(df)} bars')

# Feature engineering
df['ret_1'] = df['close'].pct_change().fillna(0)
df['ret_5'] = df['close'].pct_change(5).fillna(0)
df['sma_20'] = df['close'].rolling(20).mean().bfill()
df['ema_12'] = df['close'].ewm(span=12).mean()
df['vol_std'] = df['ret_1'].rolling(30).std().fillna(0.01)

# RSI calculation
delta = df['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
rs = gain / (loss + 1e-10)
df['rsi_14'] = 100 - (100 / (1 + rs))
df['rsi_14'] = df['rsi_14'].fillna(50)

# Bollinger Bands
df['bb_mid'] = df['close'].rolling(20).mean()
df['bb_std'] = df['close'].rolling(20).std()
df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']
df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-10)
df['bb_position'] = df['bb_position'].fillna(0.5).clip(0, 1)

# Select features
feature_cols = ['open', 'high', 'low', 'close', 'volume', 
                'ret_1', 'ret_5', 'sma_20', 'ema_12', 'vol_std',
                'rsi_14', 'bb_position']
feature_cols = [c for c in feature_cols if c in df.columns]

df = df[feature_cols].fillna(method='bfill').fillna(0)
print('Features:', feature_cols)
print('Data shape:', df.shape)

## 2) Load Data from processed_data

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install tensorflow pandas numpy matplotlib scikit-learn

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('lstm-notebook')

# GPU setup
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs found:', gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'{len(gpus)} GPU(s) configured for memory growth')
    except Exception as e:
        print('GPU setup warning:', e)
else:
    print('No GPU - running on CPU')

# Add repo to path
repo_root = Path('/Users/muaazshaikh/SuperTrader.AI').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)

## 1) Setup & GPU Configuration

# Intraday Entry/Exit Strategy - Pure LSTM
Multi-layer LSTM for sequence prediction with classification head for entry/exit signals.

**Architecture:** Stacked LSTM (64→32 units) → Dense → Sigmoid/Softmax for actions

**Key differences from DQN:**
- Direct supervised learning (no Q-values or replay buffer)
- Predicts next price movement → entry/exit signals
- Simpler, faster training
- Good for pattern recognition in time series